In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI

BASE_URL = "https://gpustack.cds.utsa.edu/v1"
API_KEY = "REPLACE-WITH-YOUR-KEY"

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

MODEL = client.models.list().data[1].id

print("Connected. Model:", MODEL)

Connected. Model: qwen3-vl-8b-instruct


In [3]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))

Hello!


In [4]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to classify a "
    "security log line and returns a dataclass with label and reason. Suggest a robust "
    "prompt format and a defensive way to parse the reply. Keep it concise."
)

print(ask_model(question))

```python
from dataclasses import dataclass
from typing import Optional

@dataclass
class TriageResult:
    label: str
    reason: str

def triage_log(line: str) -> TriageResult:
    prompt = f"""
Classify this security log line into one of: [INFO, WARN, ERROR, SUSPICIOUS, UNKNOWN].
Provide only the label and a one-sentence reason.

Log: {line}
Label: 
Reason:
    """
    # Defensive parsing: fallback to UNKNOWN if response malformed
    try:
        response = llm(prompt)
        lines = response.strip().splitlines()
        label = lines[0].strip().upper() if len(lines) > 0 else "UNKNOWN"
        reason = lines[1].strip() if len(lines) > 1 else "No reason provided."
        return TriageResult(label=label, reason=reason)
    except Exception:
        return TriageResult(label="UNKNOWN", reason="LLM failed to respond.")
```

*(Assumes `llm(prompt)` is your LLM call. Adjust parsing logic if response format differs.)*


In [5]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

In [6]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )

    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"

    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

In [7]:
TOOL_NAME = "SentinelScan"

In [8]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='Multiple failed login attempts for the root account from an external IP suggest potential brute-force attack.')


In [11]:
LOG_FILE = "security_log.csv"
MAX_ENTRIES = 20

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]

print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.csv


In [12]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")

        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")

    return path

report_path = write_report(results, TOOL_NAME)

print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# SentinelScan Security Log Assessment

Generated: 2026-11-24 14:20

Entries assessed: 20

- **BENIGN** The log line is a header row for a log format and contains no actual event data to analyze.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** Enumerating local group membership is a normal administrative task and does not indicate malicious activity.
  - `Audit Success,11/24/2026 2:16:11 PM,Microsoft-Windows-Security-Auditing,4798,User Account Management,"A user's local group membership was enumerated.`
- **BENIGN** The log line contains no actionable security event or user activity to classify as suspicious or malicious.
  - `Subject:`
- **BENIGN** The log entry originates from SYSTEM, which is a standard, non-user-initiated process and typically indicates normal system activity.
  - `Security ID:		SYSTEM`
- **SUSPICIOUS** The account name "HUSKTOP$" suggests a non-standard or possibly maliciously crafted account, often associated with at